In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Creazione e addestramento dei modelli

In [2]:
def load_and_split_data(train_path, test_path, target_col='target_assignments'):
    """Carica i dati encodati, separa feature e target, e preserva gli ID."""
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    test_ids = df_test['operator_id']
    
    X_train = df_train.drop(columns=[target_col, 'operator_id'])
    y_train = df_train[target_col]
    
    X_test = df_test.drop(columns=[target_col, 'operator_id'])
    y_test = df_test[target_col]
    
    print(f"Dataset caricati: Train ({X_train.shape[0]} righe), Test ({X_test.shape[0]} righe)")
    return X_train, y_train, X_test, y_test, test_ids

X_train, y_train, X_test, y_test, test_ids = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv')

Dataset caricati: Train (1738 righe), Test (435 righe)


In [ ]:
def prepare_asp_facts(X_test, y_test, q10, q50, q90):
    """
    Combina le predizioni con i dati di contesto e calcola lo score di incertezza.
    """
    results_df = X_test.copy()
    
    # La predizione da suggerire all'ASP arrotondata all'intero più vicino
    results_df['predicted_assignments'] = np.round(q50).astype(int)
    
    # Valore reale
    results_df['actual_assignments'] = y_test.values
    
    # Calcolo dell'Incertezza
    # Più il valore è alto, maggiore è l'incertezza del modello.
    results_df['uncertainty_score'] = q90 - q10
    
    # Estrazione del burdenScore
    burden_col = [col for col in X_test.columns if 'burdenScore' in col][0]
    
    print("\nAnteprima dei dati pronti per essere convertiti in fatti ASP:")
    display_cols = ['predicted_assignments', 'uncertainty_score', burden_col, 'density_ratio']
    display(results_df[display_cols].head())
    
    return results_df

In [4]:
def train_xgboost_quantiles(X_train, y_train, X_test):
    preds = {}
    for q, alpha in [('q10', 0.10), ('q50', 0.50), ('q90', 0.90)]:
        model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=alpha, 
                                 n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)
    return preds['q10'], preds['q50'], preds['q90']

def train_lightgbm_quantiles(X_train, y_train, X_test):
    preds = {}
    for q, alpha in [('q10', 0.10), ('q50', 0.50), ('q90', 0.90)]:
        model = LGBMRegressor(objective='quantile', alpha=alpha, 
                              n_estimators=100, random_state=42, verbose=-1)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)
    return preds['q10'], preds['q50'], preds['q90']

def train_sklearn_gb_quantiles(X_train, y_train, X_test):
    preds = {}
    for q, alpha in [('q10', 0.10), ('q50', 0.50), ('q90', 0.90)]:
        model = GradientBoostingRegressor(loss='quantile', alpha=alpha, 
                                          n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)
    return preds['q10'], preds['q50'], preds['q90']

In [5]:
def evaluate_model(model_name, y_test, q50, q10, q90):
    mae = mean_absolute_error(y_test, q50)
    rmse = np.sqrt(mean_squared_error(y_test, q50))
    r2 = r2_score(y_test, q50)
    coverage = np.mean((y_test >= q10) & (y_test <= q90))

    print(f"{model_name:<12} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R^2: {r2:.3f} | Copertura: {coverage*100:.1f}%")
    return mae

print("--- Addestramento e Valutazione Modelli ---")
models = {
    "XGBoost": train_xgboost_quantiles,
    "LightGBM": train_lightgbm_quantiles,
    "Sklearn GB": train_sklearn_gb_quantiles
}

best_mae = float('inf')
best_model_name = ""
best_predictions = None

for name, train_func in models.items():
    q10, q50, q90 = train_func(X_train, y_train, X_test)
    mae = evaluate_model(name, y_test, q50, q10, q90)
    
    # Selezione dinamica del modello con il MAE più basso
    if mae < best_mae:
        best_mae = mae
        best_model_name = name
        best_predictions = (q10, q50, q90)

print(f"\nIl modello vincitore è {best_model_name} con un MAE di {best_mae:.3f}!")

--- Addestramento e Valutazione Modelli ---
XGBoost      | MAE: 0.435 | RMSE: 0.646 | R^2: 0.740 | Copertura: 94.0%
LightGBM     | MAE: 0.400 | RMSE: 0.627 | R^2: 0.754 | Copertura: 87.4%
Sklearn GB   | MAE: 0.398 | RMSE: 0.614 | R^2: 0.764 | Copertura: 94.5%

Il modello vincitore è Sklearn GB con un MAE di 0.398!


In [6]:
def generate_clingo_facts(X_test, y_test, q10, q50, q90, original_ids, output_filename="ml_facts.lp"):
    df_facts = pd.DataFrame({
        'operator_id': original_ids,
        'predicted_assignments': np.round(q50).astype(int),
        'actual_assignments': y_test.values,
        'uncertainty_score': q90 - q10,
        'op_burdenScore': X_test['op_burdenScore'].values 
    })
    
    df_facts['ConfLevel'] = pd.qcut(df_facts['uncertainty_score'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
    
    asp_lines = [
        "% ==========================================",
        "% FATTI GENERATI DAL MACHINE LEARNING",
        f"% Modello utilizzato: {best_model_name}",
        "% Formato: ml_capacity(OP_ID, PRED_N, CONF_LEVEL).",
        "% Formato: operator_burden(OP_ID, BURDEN_SCORE).",
        "% ==========================================\n"
    ]
    
    for _, row in df_facts.iterrows():
        op_id = str(row['operator_id']).replace('.0', '')
        if op_id != '-1':
            asp_lines.append(f"ml_capacity({op_id}, {int(row['predicted_assignments'])}, {int(row['ConfLevel'])}).")
            asp_lines.append(f"operator_burden({op_id}, {int(row['op_burdenScore'])}).")
        
    with open(output_filename, 'w') as f:
        f.write("\n".join(asp_lines))
        
    print(f"File '{output_filename}' generato con {len(df_facts[df_facts['operator_id'] != -1])} record validi.")
    return df_facts

# Chiamata dinamica usando le predizioni del modello migliore
q10_best, q50_best, q90_best = best_predictions
df_facts_final = generate_clingo_facts(X_test, y_test, q10_best, q50_best, q90_best, test_ids)

display(df_facts_final.head())

File 'ml_facts.lp' generato con 435 record validi.


,operator_id,predicted_assignments,actual_assignments,uncertainty_score,op_burdenScore,ConfLevel
0,464,3,3,3.999997,2,4
1,485,4,5,1.005971,4,1
2,524,5,6,2.005779,7,2
3,226,3,3,4.003705,2,4
4,215,5,5,1.945910,2,1
